### State Schema & Data Validation with Pydantic

In this notebook, we explore how to use **Pydantic `BaseModel`** as a state schema in LangGraph.

#### Why Pydantic?
- **Runtime Validation**: Validates that all fields match their defined types during graph initialization and state transitions.
- **Type Coercion**: Automatically coerces compatible types (e.g. string `"25"` into integer `25`).
- **Validation Errors**: Raises an immediate `pydantic.ValidationError` if input data is invalid or missing required fields.
- **Attribute Access**: State in node functions can be accessed via attributes (e.g., `state.name`, `state.age`).


In [ ]:
from langgraph.graph import StateGraph, START, END
from pydantic import BaseModel, Field, ValidationError
from typing import Literal, Optional


In [ ]:
class UserState(BaseModel):
    name: str = Field(description="Name of the user")
    age: int = Field(default=0, ge=0, description="Age of the user in years (must be non-negative)")
    category: Optional[Literal["Junior", "Adult", "Senior"]] = None


#### Defining Node Functions with Pydantic State
Nodes receive an instance of the Pydantic model (`state: UserState`). They access fields using attributes (`state.age`, `state.name`) and return updates as dictionaries.


In [ ]:
def classify_age(state: UserState) -> dict:
    print(f"--- classify_age node called for {state.name} (age: {state.age}) ---")
    if state.age < 18:
        category = "Junior"
    elif state.age < 60:
        category = "Adult"
    else:
        category = "Senior"
    return {"category": category}

def welcome_node(state: UserState) -> dict:
    print(f"--- welcome node called for category: {state.category} ---")
    return {"name": f"Welcome {state.name} ({state.category})!"}


#### Building the StateGraph


In [ ]:
from IPython.display import Image, display

builder = StateGraph(UserState)

builder.add_node("classify_age", classify_age)
builder.add_node("welcome_node", welcome_node)

builder.add_edge(START, "classify_age")
builder.add_edge("classify_age", "welcome_node")
builder.add_edge("welcome_node", END)

graph = builder.compile()

try:
    display(Image(graph.get_graph().draw_mermaid_png()))
except Exception:
    print(graph.get_graph().draw_mermaid())


#### 1. Valid Input Execution
Invoking the graph with valid typed inputs.


In [ ]:
valid_result = graph.invoke({"name": "Krish", "age": 30})
print("Result with valid data:\n", valid_result)


#### 2. Automatic Type Coercion
Pydantic will automatically coerce compatible types (such as `"25"` to integer `25`).


In [ ]:
coerced_result = graph.invoke({"name": "Shivansh", "age": "25"})
print("Result with string age coerced to int:\n", coerced_result)


#### 3. Runtime Validation Error Handling
If an incompatible value (such as `"invalid_age"` for an integer field or a negative age violating `ge=0`) is passed, Pydantic immediately catches the error and raises a `ValidationError`.


In [ ]:
try:
    graph.invoke({"name": "Krish", "age": "invalid_number"})
except ValidationError as e:
    print("Caught expected Pydantic ValidationError:")
    print(e)
